In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

# 【新增】创建保存图片的文件夹
figures_dir = "figures_test"
if not os.path.exists(figures_dir):
    os.makedirs(figures_dir)
    print(f"Created directory: {figures_dir}")

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        adata = spCLUE.preprocess(adata)
        adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        
        g_spatial = spCLUE.prepare_graph(adata, "spatial", n_neighbors=6)
        g_expr = spCLUE.prepare_graph(adata, "expr", metric="euclidean", n_neighbors=8)
        graph_dict = {"spatial": g_spatial, "expr": g_expr}
        
        # 4. 模型初始化与训练
        # 原文件 Cell 8 的逻辑
        # 注意：这里将 n_clusters 参数改为动态变量，与当前切片保持一致
        spCLUE_model = spCLUE.spCLUE(adata.obsm["X_pca"], graph_dict, n_clusters,
                                     delta=0.5,          # Weight for graph-guided contrastive loss
                                    consensus_alpha=0.85, # Weight for spatial graph in consensus
                                    consensus_k=20,     # Number of consensus neighbors
                                    warmup_epochs=50,  # Epochs before enabling graph-guided loss
                                    loss_freq=5,         # 新增：每3个epoch计算一次图损失
                                    k_pos=3,             # 新增：每个anchor的正样本数
                                    k_neg=256,           # 新增：每个anchor的负样本数
                                    n_anchors=2048,      # 新增：每次采样的anchor数
                                    )
        # _, adata.obsm["spCLUE"], att_beta = spCLUE_model.train()
        _,adata.obsm["spCLUE"],_,  att_beta = spCLUE_model.train()
        
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        refinement = True
        cluster_method = "mclust"
        spCLUE.clustering(
            adata,
            n_clusters,
            key="spCLUE",
            refinement=refinement,
            cluster_methods=cluster_method,
        )
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        adata_valid = adata[adata.obs.Region.notna()]
        ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI:.8f}")
        ari_results.append(ARI)

        # 绘图：show=False 防止直接显示，便于后续保存
        adata.obs["spCLUE"] = adata.obs["mclust_refined"]
        sc.pl.spatial(
            adata, 
            color=["Region", "spCLUE"], 
            title=["Manual Annotation", f"spCLUE (ARI={round(ARI, 2)})"],
            show=False 
        )
        
        # 保存路径
        save_path = os.path.join(figures_dir, f"{sample_name}.png")
        
        # 保存图片 (bbox_inches='tight' 去除多余白边, dpi=300 保证清晰度)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        
        # 关闭当前图形，释放内存 (在循环中非常重要，否则内存会爆)
        plt.close()
        
        print(f"Figure saved to: {save_path}")
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results:
    mean_ari = np.mean(ari_results)
    median_ari = np.median(ari_results)
    print(f"ARI per slice: {[round(x, 5) for x in ari_results]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================
normalized data ---------------->
正在构建图: spatial, 使用度量: cosine ...
  -> 使用空间坐标 (euclidean)
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
spatial graph created successfully <----

正在构建图: expr, 使用度量: euclidean ...
  -> 使用 PCA 表达特征
  -> 计算最近邻 (NearestNeighbors)...
  -> 对称化与归一化...
expr graph created successfully <----

Building gated consensus graph...
   Consensus graph weight threshold (top 20%): 0.1264
   Retained 18395/91970 edges (20.0%) after filtering
✅ Gated consensus graph built (alpha=0.85, k=20, weight_threshold=0.1264)
Training Start =========================>


  4%|▍         | 21/500 [00:01<00:20, 23.75it/s]

epoch 10: 0.08437834648617268
  Batch Loss: 13.7245, Cluster Loss: 2.5637, Rec Loss: 10.3382, Contrastive Loss: 8.2264,GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1
epoch 20: 0.10628379889031496
  Batch Loss: 13.6875, Cluster Loss: 2.5535, Rec Loss: 10.3279, Contrastive Loss: 8.0610,GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1


  8%|▊         | 41/500 [00:01<00:10, 42.79it/s]

epoch 30: 0.25051772799217875
  Batch Loss: 13.6324, Cluster Loss: 2.5199, Rec Loss: 10.3184, Contrastive Loss: 7.9408,GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1
epoch 40: 0.3747297011785641
  Batch Loss: 13.5393, Cluster Loss: 2.4467, Rec Loss: 10.3088, Contrastive Loss: 7.8391,GraphGuided Loss: 0.0000,Delta: 0.0000, Beta: 1, Kappa: 0.1


...

 18%|█▊        | 91/500 [00:04<00:33, 12.20it/s]

epoch 90: 0.4074586257392877
  Batch Loss: 15.8485, Cluster Loss: 2.0203, Rec Loss: 11.9311, Contrastive Loss: 7.1666,GraphGuided Loss: 2.9512,Delta: 0.4000, Beta: 1, Kappa: 0.1


 20%|█▉        | 99/500 [00:05<00:21, 18.85it/s]

epoch 100: 0.37753830520410714
  Batch Loss: 16.0667, Cluster Loss: 1.9987, Rec Loss: 11.9330, Contrastive Loss: 7.1974,GraphGuided Loss: 2.8305,Delta: 0.5000, Beta: 1, Kappa: 0.1
fitting ...
  |                                                                      |   0%

  |======================================================================| 100%
Sample 151676 ARI: 0.45373565
Figure saved to: figures_test/151676.png

==================== Final Results ====================
ARI per slice: [0.51848, 0.5141, 0.57929, 0.37573, 0.19035, 0.32289, 0.72682, 0.77375, 0.52608, 0.45499, 0.49736, 0.45374]
Mean ARI: 0.4945
Median ARI: 0.5057
